[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/reflections/kaleidoscope.ipynb)

# The Kaleidoscope

This example constructs planar kaleidoscopes from two mirrors meeting at the origin. When the mirrors meet at a half turn divided by an integer, repeated reflections partition the plane into that many direct copies of the wedge between them and as many mirror-image copies, folding every point of the plane back into the fundamental wedge.

Each mirror through the origin is modelled as an extrinsic unit line `Plane` and each point of the plane as an intrinsic point `Point`, so reflecting across a mirror is the sandwich `mirror >> points`. Multiplying the two mirror lines, `second * first`, yields a rotor that turns by twice the angle between them and closes to a pure scalar at the dihedral order. Pairing `mirror & points` vanishes along the mirror without invoking a metric and changes sign across it, so reflecting wherever that sign is negative folds the entire plane into the wedge.

In 2D matrix notation, reflecting across a mirror with unit normal $\mathbf{u}_k = (-\sin\theta_k,\, \cos\theta_k)^{\mathsf T}$ acts on column vectors via the Householder matrix $H_k = I - 2\,\mathbf{u}_k\mathbf{u}_k^{\mathsf T}$, and two mirrors separated by $\pi / n$ generate the dihedral group $D_n$ of order $2n$, whose rotations are powers of the $2 \times 2$ matrix product $H_2 H_1 = \begin{pmatrix} \cos(2\pi/n) & -\sin(2\pi/n) \\ \sin(2\pi/n) & \cos(2\pi/n) \end{pmatrix}$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline

import numpy as np

from numga import NumpyContext
from numga.algebras import VGA2D
from examples.util.animation import show_animation
from examples.math.reflections import render

ga = VGA2D
mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()                                                      # Plane+
Point = ga.gatype.antivector()                                                  # Point-

## Mirror pairs and rotational closure

Turning the first mirror vector by a half turn minus the wedge angle makes both mirror vectors point outward from the wedge, so their inner product equals minus the cosine of the wedge angle. Multiplying the two mirrors gives the elementary rotation by twice the wedge angle, whose powers sweep through all rotational symmetries of the kaleidoscope.

In [ ]:
# Wedge orders for four side-by-side kaleidoscopes.
panel_counts = np.array([3, 4, 5, 6])
n_cases = len(panel_counts)
n_max_panels = int(panel_counts.max())

In [ ]:
first = mv.y                                                                    # [] Plane+
mirror_turns = (mv.xy * (-(np.pi - np.pi / panel_counts) / 2)).exp()            # [n_cases] Rotor+
second = mirror_turns >> first                                                  # [n_cases] Plane+
pairings = first | second                                                       # [n_cases] Scalar+
step_rotors = second * first                                                    # [n_cases] Rotor+
powers = step_rotors.broadcast_to((n_max_panels,) + step_rotors.shape).cumprod(axis=0)   # [n_max_panels, n_cases] Rotor+
full_turns = powers[panel_counts - 1, np.arange(n_cases)]                       # [n_cases] Rotor+

In [ ]:
for case, count in enumerate(panel_counts):
    print(f"{count} panels, elementary turn and {count}-th power:")
    print(step_rotors[case].coefficient_table())
    print(full_turns[case].coefficient_table())

## Folding the plane into the wedge

Points with negative pairing `mirror & points` lie outside that mirror's half-plane and reflect across it. Alternating across the two mirrors for as many rounds as the wedge order maps every point of the disk into the fundamental wedge, so symmetric points across the disk sample the same wedge location.

In [ ]:
# Pixel resolution across each circular kaleidoscope view.
n_pixels = 160

In [ ]:
def folded_plane(points: Point, mirrors: tuple[Plane, ...], n_rounds: int) -> Point:
    """Alternating reflections across `mirrors` over `n_rounds` sweeps fold `points` (`[...] Point-`) into the fundamental wedge."""
    for _ in range(n_rounds):
        for mirror in mirrors:
            side = mirror & points                                              # [...] Scalar+
            outside = (side.abs() - side) / (2 * side.abs())                    # [...] Scalar+
            points = points + outside * ((mirror >> points) - points)
    return points


horizontal, vertical = render.disk(n_pixels)
disk_points = mv.x.dual() * horizontal + mv.y.dual() * vertical                 # [n_pixels, n_pixels] Point-
folded_points = folded_plane(disk_points, (first, second[:, None, None]), n_max_panels)   # [n_cases, n_pixels, n_pixels] Point-

## Tumbling beads across four kaleidoscopes

Coloured beads orbit the origin at fixed speeds while each pixel samples the bead stack at its folded wedge coordinate. Rendering all frames takes about ten seconds.

In [ ]:
# Frame count and frame duration.
n_frames = 135
frame_duration_ms = 40

In [ ]:
show_animation(
    render.kaleidoscope_animation(folded_points, n_frames),
    "kaleidoscope", frame_duration_ms, n_frames=n_frames,
);

From left to right, the three-, four-, five- and six-fold kaleidoscopes replicate the same orbiting beads into six, eight, ten and twelve symmetric sectors around the centre.

In [ ]:
# checks
assert ((pairings + np.cos(np.pi / panel_counts)).abs() < 1e-10).all()
assert ((full_turns.select_grade(0).abs() - 1).abs() < 1e-10).all()
assert (full_turns.select_grade(2).norm() < 1e-10).all()
for mirror in (first, second[:, None, None]):
    assert ((mirror & folded_points) > -1e-10).all()